# SIFT Algorithm Experiments
Three-experiment comparison: Baseline · High Sensitivity · Speed-Optimised  
Tasks per run: two-image matching · rotation/scale invariance · homography (RANSAC) · SIFT vs ORB  
Metrics: keypoint count · inlier count · inlier ratio · reprojection error · runtime

## 0 · Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
from pathlib import Path
from dataclasses import dataclass

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print(f"OpenCV {cv2.__version__}")

## 1 · Image Loading
Place `ref.png` and `mod.png` inside a `set2/` folder next to this notebook.  
Images are downsampled to **1333 × 1000** for speed; all pixel-distance metrics are
reported at this resolution (divide by ~3 to convert back to original-pixel units).

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
REF_PATH = Path("set2/ref.png")
MOD_PATH = Path("set2/mod.png")

# Working resolution (keeps runtime manageable for large inputs)
# WORK_H, WORK_W = 1000, 1333
WORK_H, WORK_W = 3000, 4000  # uncomment for full res

def load_and_resize(path: Path, h: int = WORK_H, w: int = WORK_W):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Cannot load {path}")
    img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
    return img


img_ref = load_and_resize(REF_PATH)
img_mod = load_and_resize(MOD_PATH)
gray_ref = cv2.cvtColor(img_ref, cv2.COLOR_BGR2GRAY)
gray_mod = cv2.cvtColor(img_mod, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img, title in zip(axes, [img_ref, img_mod], ["ref.png", "mod.png"]):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=12)
    ax.axis("off")
plt.suptitle("Input images (working resolution)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print(f"Working resolution: {img_ref.shape[1]}×{img_ref.shape[0]}")

## 2 · Experiment Configurations

In [ ]:
@dataclass
class SIFTConfig:
    name: str
    color: str  # matplotlib colour for plots
    nfeatures: int  # 0 = unlimited
    nOctaveLayers: int
    contrastThreshold: float
    edgeThreshold: float
    sigma: float
    # Lowe ratio-test threshold
    lowe_ratio: float = 0.75
    # RANSAC reprojection threshold (pixels at working res)
    ransac_thresh: float = 4.0

    def build(self) -> cv2.SIFT:
        return cv2.SIFT_create(
            nfeatures=self.nfeatures,
            nOctaveLayers=self.nOctaveLayers,
            contrastThreshold=self.contrastThreshold,
            edgeThreshold=self.edgeThreshold,
            sigma=self.sigma,
        )


EXPERIMENTS = [
    SIFTConfig(
        name="Exp 1 · Baseline",
        color="#1d9e75",
        nfeatures=0,
        nOctaveLayers=3,
        contrastThreshold=0.04,
        edgeThreshold=10,
        sigma=1.6,
    ),
    SIFTConfig(
        name="Exp 2 · High sensitivity",
        color="#7f77dd",
        nfeatures=5000,
        nOctaveLayers=4,
        contrastThreshold=0.02,
        edgeThreshold=8,
        sigma=1.6,
        lowe_ratio=0.80,
    ),
    SIFTConfig(
        name="Exp 3 · Speed-optimised",
        color="#ba7517",
        nfeatures=500,
        nOctaveLayers=2,
        contrastThreshold=0.06,
        edgeThreshold=15,
        sigma=1.6,
        lowe_ratio=0.70,
    ),
]

for cfg in EXPERIMENTS:
    print(f"{cfg.name}")
    print(
        f"  nfeatures={cfg.nfeatures}, nOctaveLayers={cfg.nOctaveLayers}, "
        f"contrastThreshold={cfg.contrastThreshold}, edgeThreshold={cfg.edgeThreshold}, "
        f"lowe_ratio={cfg.lowe_ratio}"
    )

## 3 · Helper Functions

In [ ]:
def detect_and_describe(sift, gray):
    """Detect keypoints and compute SIFT descriptors."""
    t0 = time.perf_counter()
    kps, descs = sift.detectAndCompute(gray, None)
    elapsed = (time.perf_counter() - t0) * 1000  # ms
    return kps, descs, elapsed


def lowe_ratio_test(matches, ratio: float):
    """Keep only matches that pass the Lowe ratio test."""
    return [m for m, n in matches if m.distance < ratio * n.distance]


def match_sift(descs1, descs2, ratio: float):
    """FLANN-based matching with ratio test."""
    index_params = dict(algorithm=1, trees=5)  # FLANN_INDEX_KDTREE
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    raw = flann.knnMatch(descs1, descs2, k=2)
    good = lowe_ratio_test(raw, ratio)
    return good


def compute_homography(kps1, kps2, good_matches, ransac_thresh: float):
    """
    Estimate homography via RANSAC.
    Returns H, mask (or None, None if too few matches).
    """
    if len(good_matches) < 4:
        return None, None
    src_pts = np.float32([kps1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kps2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, ransac_thresh)
    return H, mask


def reprojection_error(kps1, kps2, good_matches, H, mask):
    """
    Mean reprojection error on inlier matches (pixels).
    """
    if H is None or mask is None:
        return float("nan")
    inlier_matches = [m for m, flag in zip(good_matches, mask.ravel()) if flag]
    if not inlier_matches:
        return float("nan")
    src_pts = np.float32([kps1[m.queryIdx].pt for m in inlier_matches]).reshape(
        -1, 1, 2
    )
    dst_pts = np.float32([kps2[m.trainIdx].pt for m in inlier_matches]).reshape(
        -1, 1, 2
    )
    projected = cv2.perspectiveTransform(src_pts, H)
    errors = np.linalg.norm(projected - dst_pts, axis=2).ravel()
    return float(np.mean(errors))


def draw_matches_vis(
    img1,
    kps1,
    img2,
    kps2,
    matches,
    mask=None,
    max_draw: int = 80,
    line_color=(0, 0, 255),
    line_thickness: int = 3,
    point_color=(230, 230, 230),
):
    """Return an RGB visualisation image of match lines."""
    if mask is not None:
        draw_matches = [m for m, f in zip(matches, mask.ravel()) if f]
    else:
        draw_matches = matches
    draw_matches = draw_matches[:max_draw]

    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1] = img1
    canvas[:h2, w1:w1 + w2] = img2

    for m in draw_matches:
        x1, y1 = kps1[m.queryIdx].pt
        x2, y2 = kps2[m.trainIdx].pt
        p1 = (int(round(x1)), int(round(y1)))
        p2 = (int(round(x2 + w1)), int(round(y2)))
        cv2.line(canvas, p1, p2, line_color, thickness=line_thickness, lineType=cv2.LINE_AA)
        cv2.circle(canvas, p1, 3, point_color, -1, lineType=cv2.LINE_AA)
        cv2.circle(canvas, p2, 3, point_color, -1, lineType=cv2.LINE_AA)

    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


print("Helper functions ready.")

## 4 · Task A — Two-Image Matching + Visualisation

In [ ]:
results_matching = []  # will hold dicts for final metrics table

fig, axes = plt.subplots(3, 1, figsize=(16, 18))

for ax, cfg in zip(axes, EXPERIMENTS):
    sift = cfg.build()

    kps_r, desc_r, t_r = detect_and_describe(sift, gray_ref)
    kps_m, desc_m, t_m = detect_and_describe(sift, gray_mod)
    t_detect = t_r + t_m

    t0 = time.perf_counter()
    good = match_sift(desc_r, desc_m, cfg.lowe_ratio)
    t_match = (time.perf_counter() - t0) * 1000

    H, mask = compute_homography(kps_r, kps_m, good, cfg.ransac_thresh)
    err = reprojection_error(kps_r, kps_m, good, H, mask)

    n_inliers = int(mask.sum()) if mask is not None else 0
    inlier_ratio = n_inliers / len(good) if good else 0

    results_matching.append(
        dict(
            name=cfg.name,
            kp_ref=len(kps_r),
            kp_mod=len(kps_m),
            good=len(good),
            inliers=n_inliers,
            inlier_ratio=inlier_ratio,
            reproj_err=err,
            t_detect_ms=t_detect,
            t_match_ms=t_match,
        )
    )

    vis = draw_matches_vis(img_ref, kps_r, img_mod, kps_m, good, mask)
    ax.imshow(vis)
    ax.set_title(
        f"{cfg.name}  |  kp ref={len(kps_r):,}  kp mod={len(kps_m):,}  "
        f"good={len(good)}  inliers={n_inliers}  "
        f"ratio={inlier_ratio:.2f}  reproj={err:.2f}px",
        fontsize=9,
    )
    ax.axis("off")

plt.suptitle(
    "Task A — Two-image matching (yellow lines = RANSAC inliers)", fontsize=13, y=1.0
)
plt.tight_layout()
plt.show()

## 5 · Task B — Single-Image Rotation & Scale Invariance

We synthetically transform `ref.png` and measure how many inliers SIFT recovers across a range of angles (0°–180°) and scale factors (0.4×–2.0×).

In [ ]:
def warp_image(img, angle_deg: float = 0.0, scale: float = 1.0):
    """Rotate then scale an image around its centre. Returns warped image."""
    h, w = img.shape[:2]
    cx, cy = w / 2, h / 2
    M = cv2.getRotationMatrix2D((cx, cy), angle_deg, scale)
    out = cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255),
    )
    return out, M


ANGLES = [0, 15, 30, 45, 60, 90, 120, 150, 180]
SCALES = [0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]

# ── rotation sweep ─────────────────────────────────────────────────────────
rotation_data = {cfg.name: [] for cfg in EXPERIMENTS}

for cfg in EXPERIMENTS:
    sift = cfg.build()
    kps_base, desc_base, _ = detect_and_describe(sift, gray_ref)
    for ang in ANGLES:
        warped, _ = warp_image(img_ref, angle_deg=ang)
        gray_w = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
        kps_w, desc_w, _ = detect_and_describe(sift, gray_w)
        if desc_w is None or desc_base is None:
            rotation_data[cfg.name].append(0)
            continue
        good = match_sift(desc_base, desc_w, cfg.lowe_ratio)
        _, mask = compute_homography(kps_base, kps_w, good, cfg.ransac_thresh)
        n_inliers = int(mask.sum()) if mask is not None else 0
        rotation_data[cfg.name].append(n_inliers)

# ── scale sweep ────────────────────────────────────────────────────────────
scale_data = {cfg.name: [] for cfg in EXPERIMENTS}

for cfg in EXPERIMENTS:
    sift = cfg.build()
    kps_base, desc_base, _ = detect_and_describe(sift, gray_ref)
    for sc in SCALES:
        warped, _ = warp_image(img_ref, angle_deg=0, scale=sc)
        gray_w = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
        kps_w, desc_w, _ = detect_and_describe(sift, gray_w)
        if desc_w is None or desc_base is None:
            scale_data[cfg.name].append(0)
            continue
        good = match_sift(desc_base, desc_w, cfg.lowe_ratio)
        _, mask = compute_homography(kps_base, kps_w, good, cfg.ransac_thresh)
        n_inliers = int(mask.sum()) if mask is not None else 0
        scale_data[cfg.name].append(n_inliers)

# ── plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cfg in EXPERIMENTS:
    axes[0].plot(
        ANGLES,
        rotation_data[cfg.name],
        marker="o",
        label=cfg.name,
        color=cfg.color,
        linewidth=1.8,
    )
    axes[1].plot(
        SCALES,
        scale_data[cfg.name],
        marker="s",
        label=cfg.name,
        color=cfg.color,
        linewidth=1.8,
    )

axes[0].set_xlabel("Rotation angle (°)")
axes[0].set_ylabel("RANSAC inliers")
axes[0].set_title("Rotation invariance")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Scale factor")
axes[1].set_ylabel("RANSAC inliers")
axes[1].set_title("Scale invariance")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    "Task B — Rotation & scale invariance (ref.png vs synthetic warp)", fontsize=13
)
plt.tight_layout()
plt.show()

## 6 · Task B (visual) — Sample Warped Matches

In [ ]:
DEMO_ANGLE = 45
DEMO_SCALE = 0.7

warped_ang, _ = warp_image(img_ref, angle_deg=DEMO_ANGLE)
warped_scl, _ = warp_image(img_ref, scale=DEMO_SCALE)

cases = [
    (warped_ang, f"Rotation {DEMO_ANGLE}°"),
    (warped_scl, f"Scale {DEMO_SCALE}×"),
]

for warped, label in cases:
    gray_w = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    for cfg in EXPERIMENTS:
        sift = cfg.build()
        kps_b, desc_b, _ = detect_and_describe(sift, gray_ref)
        kps_w, desc_w, _ = detect_and_describe(sift, gray_w)
        good = match_sift(desc_b, desc_w, cfg.lowe_ratio)
        H, mask = compute_homography(kps_b, kps_w, good, cfg.ransac_thresh)
        n_in = int(mask.sum()) if mask is not None else 0
        vis = draw_matches_vis(img_ref, kps_b, warped, kps_w, good, mask, max_draw=60)

        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        ax.imshow(vis)
        ax.set_title(f"{cfg.name}\n{label} | inliers={n_in}", fontsize=9)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


## 7 · Task C — Homography Estimation (RANSAC)
We overlay the warped outline of `ref.png` on `mod.png` to verify the estimated homography is geometrically correct.

In [ ]:
h_img, w_img = gray_ref.shape
corners_ref = np.float32([[0, 0], [w_img, 0], [w_img, h_img], [0, h_img]]).reshape(
    -1, 1, 2
)

for cfg in EXPERIMENTS:
    sift = cfg.build()
    kps_r, desc_r, _ = detect_and_describe(sift, gray_ref)
    kps_m, desc_m, _ = detect_and_describe(sift, gray_mod)
    good = match_sift(desc_r, desc_m, cfg.lowe_ratio)
    H, mask = compute_homography(kps_r, kps_m, good, cfg.ransac_thresh)
    err = reprojection_error(kps_r, kps_m, good, H, mask)

    canvas = cv2.cvtColor(img_mod.copy(), cv2.COLOR_BGR2RGB)

    if H is not None:
        corners_proj = cv2.perspectiveTransform(corners_ref, H)
        pts = corners_proj.reshape(-1, 2).astype(int)
        cv2.polylines(canvas, [pts], isClosed=True, color=(255, 80, 0), thickness=3)

    n_in = int(mask.sum()) if mask is not None else 0
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(canvas)
    ax.set_title(
        f"{cfg.name}\ninliers={n_in}  reproj={err:.2f}px",
        fontsize=9,
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()


## 7b · Task C (warp) — Rectified mod image


In [ ]:
h_img, w_img = gray_ref.shape

for cfg in EXPERIMENTS:
    sift = cfg.build()
    kps_r, desc_r, _ = detect_and_describe(sift, gray_ref)
    kps_m, desc_m, _ = detect_and_describe(sift, gray_mod)
    good = match_sift(desc_r, desc_m, cfg.lowe_ratio)
    H, mask = compute_homography(kps_r, kps_m, good, cfg.ransac_thresh)
    if H is None:
        print(f"{cfg.name}: homography failed")
        continue
    H_inv = np.linalg.inv(H)

    rectified = cv2.warpPerspective(
        img_mod,
        H_inv,
        (w_img, h_img),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255),
    )

    n_in = int(mask.sum()) if mask is not None else 0
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(img_ref, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Reference (ref.png)", fontsize=9)
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(rectified, cv2.COLOR_BGR2RGB))
    axes[1].set_title(
        f"{cfg.name} rectified mod\ninliers={n_in}",
        fontsize=9,
    )
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


## 8 · Task D — SIFT vs ORB Comparison

In [ ]:
def run_orb(gray1, gray2, n_features: int = 1000, ransac_thresh: float = 4.0):
    """Detect, match, and compute homography with ORB + BFMatcher Hamming."""
    orb = cv2.ORB_create(nfeatures=n_features)
    t0 = time.perf_counter()
    kps1, desc1 = orb.detectAndCompute(gray1, None)
    kps2, desc2 = orb.detectAndCompute(gray2, None)
    t_detect = (time.perf_counter() - t0) * 1000

    if desc1 is None or desc2 is None:
        return dict(
            kp1=0,
            kp2=0,
            good=0,
            inliers=0,
            inlier_ratio=0,
            reproj_err=float("nan"),
            t_detect_ms=t_detect,
            t_match_ms=0,
        )

    t0 = time.perf_counter()
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    raw = bf.knnMatch(desc1, desc2, k=2)
    good = [m for m, n in raw if m.distance < 0.75 * n.distance]
    t_match = (time.perf_counter() - t0) * 1000

    H, mask = compute_homography(kps1, kps2, good, ransac_thresh)
    err = reprojection_error(kps1, kps2, good, H, mask)
    n_in = int(mask.sum()) if mask is not None else 0

    return dict(
        kp1=len(kps1),
        kp2=len(kps2),
        good=len(good),
        inliers=n_in,
        inlier_ratio=n_in / len(good) if good else 0,
        reproj_err=err,
        t_detect_ms=t_detect,
        t_match_ms=t_match,
    )


# Run ORB at three feature counts matching the SIFT experiments roughly
ORB_N = [2000, 5000, 500]  # ORB default = 500; scale up to be fair to Exp1/2
orb_results = []

for cfg, n_orb in zip(EXPERIMENTS, ORB_N):
    r = run_orb(gray_ref, gray_mod, n_features=n_orb)
    r["exp_name"] = cfg.name
    r["orb_n"] = n_orb
    orb_results.append(r)

# ── bar chart comparison ───────────────────────────────────────────────────
metrics_compare = ["inliers", "inlier_ratio", "reproj_err"]
titles_compare = ["RANSAC inliers", "Inlier ratio", "Reprojection error (px)"]
x = np.arange(len(EXPERIMENTS))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(axes, metrics_compare, titles_compare):
    sift_vals = [r[metric] for r in results_matching]
    orb_vals = [r[metric] for r in orb_results]

    bars_s = ax.bar(
        x - width / 2,
        sift_vals,
        width,
        label="SIFT",
        color=[cfg.color for cfg in EXPERIMENTS],
        alpha=0.85,
    )
    bars_o = ax.bar(
        x + width / 2, orb_vals, width, label="ORB", color="#888780", alpha=0.75
    )

    ax.set_xticks(x)
    ax.set_xticklabels([f"E{i + 1}" for i in range(3)], fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    # value labels on bars
    for bar in list(bars_s) + list(bars_o):
        h = bar.get_height()
        label = f"{h:.2f}" if metric in ("inlier_ratio", "reproj_err") else str(int(h))
        ax.annotate(
            label,
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7.5,
        )

plt.suptitle(
    "Task D — SIFT vs ORB (E1=Baseline, E2=High sensitivity, E3=Speed-opt.)",
    fontsize=12,
    y=1.02,
)
plt.tight_layout()
plt.show()

## 9 · Runtime Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

exp_labels = [f"E{i + 1}" for i in range(3)]
sift_detect = [r["t_detect_ms"] for r in results_matching]
sift_match = [r["t_match_ms"] for r in results_matching]
orb_detect = [r["t_detect_ms"] for r in orb_results]
orb_match = [r["t_match_ms"] for r in orb_results]

x = np.arange(3)
w = 0.35

# Detection time
ax = axes[0]
ax.bar(
    x - w / 2,
    sift_detect,
    w,
    label="SIFT detect",
    color=[cfg.color for cfg in EXPERIMENTS],
    alpha=0.85,
)
ax.bar(x + w / 2, orb_detect, w, label="ORB detect", color="#888780", alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(exp_labels)
ax.set_ylabel("ms")
ax.set_title("Detection time (both images combined)")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# Matching time
ax = axes[1]
ax.bar(
    x - w / 2,
    sift_match,
    w,
    label="SIFT match",
    color=[cfg.color for cfg in EXPERIMENTS],
    alpha=0.85,
)
ax.bar(x + w / 2, orb_match, w, label="ORB match", color="#888780", alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(exp_labels)
ax.set_ylabel("ms")
ax.set_title("Matching time")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

plt.suptitle("Runtime comparison — SIFT vs ORB", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 10 · Summary Metrics Table

In [ ]:
import pandas as pd

rows = []
for i, (cfg, sr, orb) in enumerate(zip(EXPERIMENTS, results_matching, orb_results)):
    rows.append(
        {
            "Experiment": f"E{i + 1}",
            "Config": cfg.name,
            "SIFT kp (ref)": f"{sr['kp_ref']:,}",
            "SIFT kp (mod)": f"{sr['kp_mod']:,}",
            "SIFT good matches": sr["good"],
            "SIFT inliers": sr["inliers"],
            "SIFT inlier %": f"{sr['inlier_ratio'] * 100:.1f}%",
            "SIFT reproj (px)": f"{sr['reproj_err']:.3f}",
            "SIFT detect (ms)": f"{sr['t_detect_ms']:.1f}",
            "ORB inliers": orb["inliers"],
            "ORB inlier %": f"{orb['inlier_ratio'] * 100:.1f}%",
            "ORB reproj (px)": f"{orb['reproj_err']:.3f}",
            "ORB detect (ms)": f"{orb['t_detect_ms']:.1f}",
        }
    )

df = pd.DataFrame(rows).set_index("Experiment")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(df)

# Plain-text summary
print("\n── KEY TAKEAWAYS ────────────────────────────────────────────────────")
for i, (cfg, sr) in enumerate(zip(EXPERIMENTS, results_matching)):
    print(
        f"  E{i + 1} {cfg.name}: {sr['inliers']} inliers, "
        f"reproj={sr['reproj_err']:.3f}px, "
        f"detect={sr['t_detect_ms']:.1f}ms"
    )

## 11 · Keypoint Heatmaps (spatial distribution)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, cfg in zip(axes, EXPERIMENTS):
    sift = cfg.build()
    kps, _, _ = detect_and_describe(sift, gray_ref)

    # Accumulate keypoint positions into a density image
    density = np.zeros(gray_ref.shape, dtype=np.float32)
    for kp in kps:
        x_i, y_i = int(kp.pt[0]), int(kp.pt[1])
        if 0 <= y_i < density.shape[0] and 0 <= x_i < density.shape[1]:
            density[y_i, x_i] += 1

    # Blur for readability
    density = cv2.GaussianBlur(density, (25, 25), 0)
    density = density / density.max() if density.max() > 0 else density

    # Composite with ref image
    base = cv2.cvtColor(img_ref, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    heatmap = cv2.applyColorMap((density * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    composite = cv2.addWeighted(base, 0.55, heatmap, 0.45, 0)

    ax.imshow(np.clip(composite, 0, 1))
    ax.set_title(f"{cfg.name}\n{len(kps):,} keypoints", fontsize=9)
    ax.axis("off")

plt.suptitle("Keypoint spatial density heatmap — ref.png", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()